# 12 — Aircraft Rotation: modelling delay propagation

**Airline Operations Intelligence Platform** · Notebook 12 · *runs locally*

## The observation that motivates this notebook

Notebook 05 measured that **late aircraft is the single largest cause of delay minutes
(39.84%)** — larger than carrier, NAS and weather. Yet the model in notebook 06 cannot see
it, because it treats every flight as an independent row.

Flights are not independent. An aircraft flies 4–6 legs a day, and a delay on leg 1
propagates down the chain. That structure is **already in the data**: `tail_number`
identifies the airframe, and the scheduled times order its legs.

## The honest catch: this changes the prediction horizon

Notebook 06 answers: *"given the schedule, how risky is this flight?"* — answerable weeks
ahead, at booking or planning time.

Adding the inbound aircraft's actual arrival delay answers a different question:
*"it is now two hours before departure and the inbound is 40 minutes late — how risky is
this flight?"* — a **day-of operational** question.

Both are legitimate. They are not comparable as "the same model, improved", and this
notebook reports them as two distinct models rather than pretending the second is simply a
better version of the first.

| Model | Horizon | Uses inbound status |
|---|---|---|
| **Planning** (notebook 06) | Weeks ahead | No |
| **Day-of** (this notebook) | Hours ahead | Yes |

In [ ]:
import sys, time, json
sys.path.insert(0, "../src")

from config import build_spark, PATHS
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

spark = build_spark("12-rotation", **{"spark.driver.memory": "5g"})

src = PATHS["curated"] / "flights_weather.parquet"
flights = spark.read.parquet(str(src))
print(f"Source: {src.name}   rows: {flights.count():,}")

---
## 1. Building the rotation chain

For each aircraft on each day, order its legs by scheduled departure and look backwards
one step. `lag()` over a window partitioned by `(tail_number, flight_date)` gives:

- **`prev_arr_delay`** — how late the inbound aircraft actually arrived
- **`prev_origin`** — where it came from
- **`scheduled_turnaround`** — minutes between the inbound's scheduled arrival and this
  flight's scheduled departure. A tight turnaround absorbs less upstream delay.
- **`leg_number`** — position in the day's chain; delay accumulates down it

`tail_number` is null for 0.25% of flights (notebook 01). Those cannot be chained and are
flagged rather than dropped.

In [ ]:
w = Window.partitionBy("tail_number", "flight_date").orderBy("sched_dep_min")

rot = (flights
    .withColumn("leg_number", F.when(F.col("tail_number").isNull(), None)
                               .otherwise(F.row_number().over(w)))
    .withColumn("prev_arr_delay",  F.lag("arr_delay").over(w))
    .withColumn("prev_sched_arr",  F.lag("sched_arr_min").over(w))
    .withColumn("prev_origin",     F.lag("origin").over(w))
    .withColumn("prev_status",     F.lag("status").over(w))
    .withColumn("scheduled_turnaround",
                F.when(F.col("prev_sched_arr").isNull(), None)
                 .otherwise(F.col("sched_dep_min") - F.col("prev_sched_arr")))
    .withColumn("has_inbound", F.col("prev_arr_delay").isNotNull().cast("int")))

rot = rot.cache()
total = rot.count()
with_inbound = rot.filter(F.col("has_inbound") == 1).count()

print(f"Flights total            : {total:,}")
print(f"With a traceable inbound : {with_inbound:,}  ({100*with_inbound/total:.1f}%)")
print(f"First leg of the day     : {rot.filter(F.col('leg_number') == 1).count():,}")
print(f"No tail number           : {rot.filter(F.col('tail_number').isNull()).count():,}")

In [ ]:
rot.filter(F.col("has_inbound") == 1).select(
    "tail_number", "flight_date", "leg_number", "prev_origin", "origin", "destination",
    "prev_arr_delay", "scheduled_turnaround", "arr_delay").orderBy(
    "tail_number", "flight_date", "leg_number").show(8, truncate=False)

---
## 2. Does delay actually propagate?

Before modelling it, measure it. If the inbound's delay does not relate to this flight's
delay, the feature is worthless regardless of how principled it looks.

In [ ]:
comp = rot.filter((F.col("status") == "completed") & (F.col("has_inbound") == 1))
base_rate = rot.filter(F.col("status") == "completed").agg(F.avg("is_delayed")).first()[0]

print(f"Network delay rate: {100*base_rate:.2f}%\n")
print(f"{'INBOUND ARRIVED':<24}{'FLIGHTS':>12}{'DELAY RATE':>12}{'vs BASE':>11}")
print("-" * 60)
bands = [("early / on time", None, 0), ("0-15 min late", 0, 15),
         ("15-30 min late", 15, 30), ("30-60 min late", 30, 60),
         ("60-120 min late", 60, 120), ("2+ hours late", 120, None)]
for label, lo, hi in bands:
    q = comp
    if lo is not None: q = q.filter(F.col("prev_arr_delay") >= lo)
    if hi is not None: q = q.filter(F.col("prev_arr_delay") < hi)
    n = q.count()
    if n < 100: continue
    r = q.agg(F.avg("is_delayed")).first()[0]
    print(f"{label:<24}{n:>12,}{100*r:>11.2f}%{100*(r-base_rate):>+10.2f}pp")

In [ ]:
# Correlation between the inbound's delay and this flight's.
corr = comp.stat.corr("prev_arr_delay", "arr_delay")
print(f"Pearson correlation, inbound arrival delay vs this flight's arrival delay: {corr:.4f}")

print("\nDelay rate by scheduled turnaround time:")
(comp.filter(F.col("scheduled_turnaround").between(0, 600))
     .withColumn("turn_band",
        F.when(F.col("scheduled_turnaround") < 45, "1. under 45 min")
         .when(F.col("scheduled_turnaround") < 90, "2. 45-90 min")
         .when(F.col("scheduled_turnaround") < 180, "3. 90-180 min")
         .otherwise("4. over 3 hours"))
     .groupBy("turn_band")
     .agg(F.count("*").alias("flights"),
          F.round(100*F.avg("is_delayed"), 2).alias("delay_rate_pct"))
     .orderBy("turn_band").show(truncate=False))

In [ ]:
print("Delay rate by position in the aircraft's daily chain:")
(comp.filter(F.col("leg_number") <= 8)
     .groupBy("leg_number")
     .agg(F.count("*").alias("flights"),
          F.round(100*F.avg("is_delayed"), 2).alias("delay_rate_pct"),
          F.round(F.avg("arr_delay"), 2).alias("avg_arr_delay"))
     .orderBy("leg_number").show(truncate=False))
print("Delay compounds down the chain -- each leg inherits the accumulated lateness")
print("of the ones before it. This is the 39.84% 'late aircraft' cause, made visible.")

---
## 3. Train the day-of model

Same split strategy, same evaluation harness and the tuned hyperparameters found in
notebook 06, so the only difference is the feature set.

In [ ]:
CATEGORICAL = ["airline_code", "time_of_day", "season"]
BASE_NUMERIC = ["month", "day_of_week", "sched_dep_hour", "distance", "sched_duration",
                "is_weekend_int", "origin_delay_rate", "dest_delay_rate",
                "airline_delay_rate", "route_delay_rate",
                "origin_hour_delay_rate", "airline_origin_delay_rate",
                "temp_c", "dewpoint_c", "wind_speed", "visibility_m", "ceiling_m",
                "precip_mm", "wx_thunderstorm", "wx_snow", "wx_rain", "wx_fog",
                "wx_freezing", "wx_haze_smoke"]
ROTATION = ["prev_arr_delay", "scheduled_turnaround", "leg_number", "has_inbound"]

# prev_arr_delay is the previous flight's outcome, known before THIS flight departs.
# It is therefore legitimate for a day-of model and inadmissible for a planning one.
print("Rotation features:", ROTATION)

In [ ]:
cols = ["is_delayed", "airline_code", "origin", "destination", "route", "month",
        "day_of_week", "sched_dep_hour", "distance", "sched_duration", "time_of_day",
        "season", "flight_date"] + [c for c in BASE_NUMERIC if c.startswith(("temp","dew","wind","vis","ceil","precip","wx_"))] + ROTATION

base = (rot.filter(F.col("status") == "completed")
           .select(*cols, F.col("is_weekend").cast("int").alias("is_weekend_int"))
           .filter(F.col("is_delayed").isNotNull())
           .withColumn("row_id", F.monotonically_increasing_id())
           .persist(StorageLevel.MEMORY_AND_DISK))
n_base = base.count()

# `rot` caches all 61 columns of every flight and is dead the moment `base` is
# materialised. Releasing it here returns that memory to the two GBT fits, which are
# the only thing in this notebook that is actually memory-hungry.
rot.unpersist()

SEED = 42
train = base.sampleBy("is_delayed", {0: 0.8, 1: 0.8}, seed=SEED).persist(StorageLevel.MEMORY_AND_DISK)
test  = base.join(train.select("row_id"), "row_id", "left_anti").persist(StorageLevel.MEMORY_AND_DISK)
n_train, n_test = train.count(), test.count()
GLOBAL_RATE = train.agg(F.avg("is_delayed")).first()[0]
print(f"Train {n_train:,}   Test {n_test:,}   positive {100*GLOBAL_RATE:.2f}%")
assert n_train + n_test == n_base

In [ ]:
SMOOTHING = 100
def smoothed(keys, out_col):
    return (train.groupBy(*keys).agg(F.count("*").alias("n"), F.avg("is_delayed").alias("r"))
            .withColumn(out_col, (F.col("n")*F.col("r") + F.lit(SMOOTHING*GLOBAL_RATE))
                                 / (F.col("n")+F.lit(SMOOTHING)))
            .select(*keys, out_col))

R = {"o": smoothed(["origin"], "origin_delay_rate"),
     "d": smoothed(["destination"], "dest_delay_rate"),
     "a": smoothed(["airline_code"], "airline_delay_rate"),
     "r": smoothed(["route"], "route_delay_rate"),
     "oh": smoothed(["origin","sched_dep_hour"], "origin_hour_delay_rate"),
     "ao": smoothed(["airline_code","origin"], "airline_origin_delay_rate")}

def add(df):
    out = (df.join(F.broadcast(R["o"]), "origin", "left")
             .join(F.broadcast(R["d"]), "destination", "left")
             .join(F.broadcast(R["a"]), "airline_code", "left")
             .join(F.broadcast(R["r"]), "route", "left")
             .join(F.broadcast(R["oh"]), ["origin","sched_dep_hour"], "left")
             .join(F.broadcast(R["ao"]), ["airline_code","origin"], "left"))
    fills = {c: GLOBAL_RATE for c in
             ["origin_delay_rate","dest_delay_rate","airline_delay_rate",
              "route_delay_rate","origin_hour_delay_rate","airline_origin_delay_rate"]}
    # No inbound -> neutral values, with has_inbound telling the model which case it is.
    fills.update({"prev_arr_delay": 0.0, "scheduled_turnaround": 120.0, "leg_number": 1.0})
    wx = [c for c in df.columns if c.startswith(("temp","dew","wind","vis","ceil","precip","wx_"))]
    if wx:
        means = df.select([F.avg(c).alias(c) for c in wx]).first().asDict()
        fills.update({k: (v if v is not None else 0.0) for k, v in means.items()})
    return out.fillna(fills)

train_f = add(train).persist(StorageLevel.MEMORY_AND_DISK)
test_f  = add(test).persist(StorageLevel.MEMORY_AND_DISK)
print(f"train_f {train_f.count():,}   test_f {test_f.count():,}")

In [ ]:
RESULTS = []

def run(numeric, name):
    w_pos = (1 - GLOBAL_RATE) / GLOBAL_RATE
    tr = train_f.withColumn("weight", F.when(F.col("is_delayed")==1, w_pos).otherwise(1.0))
    idx = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in CATEGORICAL]
    enc = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_vec", handleInvalid="keep") for c in CATEGORICAL]
    asm = VectorAssembler(inputCols=[f"{c}_vec" for c in CATEGORICAL] + numeric,
                          outputCol="features", handleInvalid="skip")
    prep = Pipeline(stages=idx+enc+[asm]).fit(tr)
    trv = prep.transform(tr).select("features","is_delayed","weight").persist(StorageLevel.MEMORY_AND_DISK)
    tev = prep.transform(test_f).select("features","is_delayed").persist(StorageLevel.MEMORY_AND_DISK)
    trv.count(); tev.count()

    t0 = time.time()
    model = GBTClassifier(labelCol="is_delayed", featuresCol="features", weightCol="weight",
                          maxDepth=8, maxIter=80, maxBins=64, seed=SEED,
                          subsamplingRate=0.7).fit(trv)
    secs = time.time() - t0

    preds = model.transform(tev)
    auc = BinaryClassificationEvaluator(labelCol="is_delayed", rawPredictionCol="rawPrediction",
                                        metricName="areaUnderROC").evaluate(preds)
    scored = preds.select("is_delayed", vector_to_array("probability").getItem(1).alias("p")) \
                  .persist(StorageLevel.MEMORY_AND_DISK)
    pos, tot = scored.filter("is_delayed = 1").count(), scored.count()
    best = None
    for t in [x/100 for x in range(25, 76, 5)]:
        tp = scored.filter((F.col("p")>=t) & (F.col("is_delayed")==1)).count()
        fp = scored.filter((F.col("p")>=t) & (F.col("is_delayed")==0)).count()
        fn = pos - tp; tn = tot - tp - fp - fn
        pr = tp/(tp+fp) if tp+fp else 0; rc = tp/(tp+fn) if tp+fn else 0
        f1 = 2*pr*rc/(pr+rc) if pr+rc else 0
        if best is None or f1 > best["f1"]:
            best = dict(threshold=round(t,2), precision=pr, recall=rc, f1=f1,
                        accuracy=(tp+tn)/tot, tp=tp, fp=fp, fn=fn, tn=tn)
    scored.unpersist(); trv.unpersist(); tev.unpersist()

    row = dict(model=name, roc_auc=round(auc,4), f1=round(best["f1"],4),
               precision=round(best["precision"],4), recall=round(best["recall"],4),
               accuracy=round(best["accuracy"],4), threshold=best["threshold"],
               train_seconds=round(secs,1))
    RESULTS.append((row, model, prep))
    print(f"\n{name}")
    print(f"  ROC-AUC {auc:.4f} | F1 {best['f1']:.4f} | precision {best['precision']:.4f} "
          f"| recall {best['recall']:.4f} | threshold {best['threshold']} | {secs:.0f}s")
    return row

In [ ]:
r_plan = run(BASE_NUMERIC, "Planning model (no rotation) -- weeks ahead")

In [ ]:
r_dayof = run(BASE_NUMERIC + ROTATION, "Day-of model (with rotation) -- hours ahead")

---
## 4. Comparison

In [ ]:
print(f"{'MODEL':<48}{'ROC-AUC':>9}{'F1':>8}{'RECALL':>8}")
print("-" * 73)
for row, _, _ in RESULTS:
    print(f"{row['model']:<48}{row['roc_auc']:>9.4f}{row['f1']:>8.4f}{row['recall']:>8.4f}")
print("-" * 73)
d_auc = r_dayof["roc_auc"] - r_plan["roc_auc"]
d_f1  = r_dayof["f1"] - r_plan["f1"]
print(f"{'Gain from rotation features':<48}{d_auc:>+9.4f}{d_f1:>+8.4f}")

In [ ]:
# Which rotation feature carries the signal?
row, model, prep = RESULTS[-1]
attrs = prep.transform(train_f.withColumn("weight", F.lit(1.0))).limit(5) \
            .schema["features"].metadata["ml_attr"]["attrs"]
names = {}
for kind in ("numeric","binary","nominal"):
    for a in attrs.get(kind, []):
        names[a["idx"]] = a["name"]
imp = model.featureImportances.toArray()
pairs = sorted(((names[i], v) for i, v in enumerate(imp)), key=lambda kv: -kv[1])

print(f"{'FEATURE':<32}{'IMPORTANCE':>11}")
print("-"*46)
for nm, v in pairs[:12]:
    mark = "  <- rotation" if nm in ROTATION else ""
    print(f"{nm:<32}{v:>11.4f}{mark}")

rot_share = sum(v for nm, v in pairs if nm in ROTATION)
print(f"\nRotation features account for {100*rot_share:.1f}% of total importance.")

---
## 5. What this means

**The gain is real but bounded.** Knowing the inbound aircraft is late is genuinely
informative — the propagation measured in §2 is unambiguous — but it does not transform the
problem. Most delayed flights are delayed for reasons other than a late inbound, and most
late inbounds still produce an on-time departure because schedules build in turnaround slack.

**The two models answer different questions and should not be merged.** A booking site
needs the planning model; an airline's day-of operations desk needs this one. Reporting the
day-of number as "the model improved" would be quietly changing the question to make the
answer look better.

**What is still missing.** Even here the model sees only the *previous* leg. A full
treatment would propagate delay through the whole chain (a graph problem), and would include
crew rotations, which the dataset does not contain at all.

In [ ]:
out = PATHS["marts"] / "ml_rotation_results.parquet"
spark.createDataFrame([r for r, _, _ in RESULTS]).coalesce(1) \
     .write.mode("overwrite").parquet(str(out))

payload = {"models": [r for r, _, _ in RESULTS],
           "rotation_features": ROTATION,
           "rotation_importance_share": round(rot_share, 4),
           "inbound_traceable_pct": round(100*with_inbound/total, 2),
           "delay_correlation_prev_vs_current": round(corr, 4)}
(PATHS["marts"] / "ml_rotation_results.json").write_text(json.dumps(payload, indent=2))
print("Wrote ml_rotation_results.{parquet,json}")

RESULTS[-1][1].write().overwrite().save(str(PATHS["models"] / "dayof_delay_classifier"))
RESULTS[-1][2].write().overwrite().save(str(PATHS["models"] / "dayof_feature_pipeline"))
print("Saved day-of model.")

In [ ]:
for df in (rot, base, train, test, train_f, test_f):
    try: df.unpersist()
    except Exception: pass
spark.stop()
print("Notebook 12 complete.")